# PRAGMA Pre-training

This notebook demonstrates the self-supervised pre-training loop for the PRAGMA model. We use Masked Event Prediction, where a percentage of transactions in a sequence are masked, and the model must reconstruct their features.

In [ ]:
import torch
import torch.nn as nn
import sys
sys.path.append('../src')

from pragma_model import PRAGMA

# Mock configuration for pre-training demonstration
profile_config = {
    'num_numerical_features': 10,
    'num_categorical_features': 5,
    'cat_embedding_dims': [(100, 8), (50, 4), (20, 4), (10, 2), (5, 2)]
}

event_config = {
    'event_dim': 20, # Dimensionality of a single event
    'embed_dim': 64,
    'num_heads': 4,
    'num_layers': 3,
    'max_seq_len': 100
}

model = PRAGMA(profile_config, event_config, embed_dim=64)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

In [ ]:
# Optimization
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss() # Using MSE for continuous event reconstruction

## Pre-training Loop

In [ ]:
def pretrain_step(model, optimizer, criterion, batch, mask_prob=0.15):
    model.train()
    
    x_num, x_cat, events, seq_lengths = batch
    
    # Create Masked Events (copy original events for labels)
    labels = events.clone()
    
    # Generate mask
    mask = torch.rand(events.shape[:2], device=events.device) < mask_prob
    
    # Don't mask padding tokens
    pad_mask = torch.arange(events.shape[1], device=events.device)[None, :] >= seq_lengths[:, None]
    mask = mask & ~pad_mask
    
    # Apply mask (e.g., zero out masked events)
    masked_events = events.clone()
    masked_events[mask] = 0.0
    
    # Forward pass
    optimizer.zero_grad()
    fused, mlm_preds = model(x_num, x_cat, masked_events, seq_lengths, pretrain=True)
    
    # Compute loss only on masked positions
    loss = criterion(mlm_preds[mask], labels[mask])
    
    loss.backward()
    optimizer.step()
    
    return loss.item()

# Example dummy batch
batch_size = 32
seq_len = 50

x_num = torch.randn(batch_size, 10).to(device)
x_cat = torch.randint(0, 5, (batch_size, 5)).to(device)
events = torch.randn(batch_size, seq_len, 20).to(device)
seq_lengths = torch.randint(10, seq_len, (batch_size,)).to(device)

batch = (x_num, x_cat, events, seq_lengths)

# Run a few steps
for i in range(10):
    loss = pretrain_step(model, optimizer, criterion, batch)
    print(f"Step {i+1}, Loss: {loss:.4f}")

In [ ]:
# Save pre-trained checkpoint
torch.save(model.state_dict(), '../models/pragma_pretrained.pth')
print("Pre-trained model saved to ../models/pragma_pretrained.pth")